# FIN-01 — Automatic Transaction Categorization

**Capstone project — FinTech track**

Client problem: a digital banking app shows raw, inconsistent merchant
transaction descriptions. Manual and rule-based categorization doesn't
scale. This notebook builds, evaluates, and exports an ML pipeline that
assigns each transaction to a spending category.

See `README.md` for the project structure and `docs/limitations_and_risks.md`
for the full design-decision writeup.

## 1. Setup & data download

In [1]:
import os, re, json, time
import urllib.request
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
import joblib

DATA_URL = "https://raw.githubusercontent.com/utribedi/Bank_transaction_category_predictor/main/Training.xlsx"
DATA_PATH = "data/Training.xlsx"

os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_excel(DATA_PATH)
print(df.shape)
df.head()

(40000, 14)


,sor,cdf_seq_no,trans_desc,merchant_cat_code,amt,db_cr_cd,payment_reporting_category,payment_category,is_international,default_brand,default_location,qrated_brand,coalesced_brand,Category
0,HH,T20110701260061756,RECUR DEBIT CRD PMT11/11 DELTA DENTAL OF A11 O...,6300.0,58.34,D,Card,Debit Card,False,DELTA DENTAL OF A11 OF,111-1111111 AR,Delta Dental,Delta Dental,Finance
1,HH,T201302289909010349,"CHECK CRD PURCHASE 11/11 SURETY SOLUTIONS, ...",NaN,103.00,D,Card,Check Card,False,"SURETY SOLUTIONS,",111-111-1111 OR,Surety Solutions,Surety Solutions,Finance
2,HH,T20130726991361190114550,CHECK CRD PURCHASE 11/11 THE COPY STOP ...,NaN,22.44,D,Card,Check Card,False,THE COPY STOP,SALT LAKE CIT UT,The Copy Stop,The Copy Stop,Finance
3,BK,T201207095780929968,MARKET ALERT INC 111-111-1111 TX,7375.0,22.44,NaN,Card,Credit Card,False,MARKET ALERT INC,111-111-1111 TX,Market Alert,Market Alert,Finance
4,HH,T20131230990558080004939,CHECK CRD PURCHASE 11/11 PERT- NER PERFECT ...,NaN,66.25,D,Card,Check Card,False,PERT- NER PERFECT,THE WOODLANDS TX,Hometown Insurance Partners,Hometown Insurance Partners,Finance


## 2. Exploratory data analysis

In [2]:
print("Columns:", list(df.columns))
print()
print("Category distribution:")
print(df['Category'].value_counts())
print()
print("Missing values:")
print(df.isnull().sum())

Columns: ['sor', 'cdf_seq_no', 'trans_desc', 'merchant_cat_code', 'amt', 'db_cr_cd', 'payment_reporting_category', 'payment_category', 'is_international', 'default_brand', 'default_location', 'qrated_brand', 'coalesced_brand', 'Category']

Category distribution:
Category
Retail Trade                                 13500
Entertainment                                11255
Trade, Professional and Personal Services     5275
Health and Community Services                 4157
Services to Transport                         2317
Travel                                        1489
Property and Business Services                1095
Education                                      445
Communication Services                         282
Finance                                        185
Name: count, dtype: int64

Missing values:
sor                               0
cdf_seq_no                        0
trans_desc                        0
merchant_cat_code             15309
amt                            

**Observations:**
- 40,000 rows, 10 categories, severe imbalance (185 to 13,500 examples per category)
- `trans_desc` is the primary text signal but contains heavy boilerplate
  (dates, masked card digits, placeholder location codes)
- `coalesced_brand` has zero missing values and often contains a cleaner
  merchant name than the raw description
- `merchant_cat_code` is ~38% missing — excluded from this iteration
  (see limitations doc)

In [3]:
sample = df.sample(5, random_state=1)[['trans_desc','coalesced_brand','Category']]
for _, row in sample.iterrows():
    print(row['trans_desc'])
    print('  brand:', row['coalesced_brand'], '| category:', row['Category'])
    print()

CHECK CRD PURCHASE 11/11 LEATHERGOODS CONNECTIO  111-111-1111  GA 111111XXXXXX1111 111111111111111                               ?MCC=1111 11
  brand: Leather Goods Connection | category: Retail Trade

CHECK CRD PURCHASE 11/11 SCRAP APPLE QUILTS      ST GEORGE     UT 111111XXXXXX1111 111111111111111                               ?MCC=1111
  brand: Scrap Apple Quilts | category: Retail Trade

CHECK CRD PURCHASE 11/11 HEMET ASIAN MARKET LLC  HEMET         CA 111111XXXXXX1111 111111111111111                               ?MCC=1111 11
  brand: Hemet Asian Market | category: Retail Trade

CHECK CRD PURCHASE 11/11 GOONY GOLF OF SPRING L  SPRING LAKE P MN 111111XXXXXX1111 111111111111111                               ?MCC=1111 11
  brand: Spring Lake Park Goony Golf Mini Golf | category: Entertainment

CHECK CRD PURCHASE 11/11 Patty's Place           111-1111111   PA 111111XXXXXX1111 111111111111111                               ?MCC=1111
  brand: Pattys Place | category: Entertainment



## 3. Text cleaning

In [4]:
def clean_text(s):
    s = str(s).upper()
    s = re.sub(r'CHECK\s*CRD\s*PURCHASE', ' ', s)
    s = re.sub(r'HSA\s*CARD\s*PURCHASE', ' ', s)
    s = re.sub(r'RECURRING\s*PMT', ' ', s)
    s = re.sub(r'\bMCC\b', ' ', s)
    s = re.sub(r'[^A-Z\s]', ' ', s)
    s = re.sub(r'\bX{3,}\b', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df['text'] = df['trans_desc'].astype(str) + ' ' + df['coalesced_brand'].astype(str)
df['clean_text'] = df['text'].apply(clean_text)

# before / after example
print("BEFORE:", df['text'].iloc[0])
print("AFTER: ", df['clean_text'].iloc[0])

BEFORE: RECUR DEBIT CRD PMT11/11 DELTA DENTAL OF A11 OF  111-1111111   AR 111111XXXXXX1111 111111111111111                               ?MCC=1111 11 Delta Dental
AFTER:  RECUR DEBIT CRD PMT DELTA DENTAL OF A OF AR DELTA DENTAL


## 4. Category taxonomy decision

Three categories (Communication Services, Finance, Education) have fewer
than 300 training examples each. Training on all 10 categories with
`class_weight='balanced'` gives these classes decent recall but very poor
precision (0.12–0.17), because the balancing makes the model over-predict
them. We merge these three into a single "Other Services" bucket.

In [5]:
RARE_CLASS_MERGE = {
    'Communication Services': 'Other Services',
    'Finance': 'Other Services',
    'Education': 'Other Services',
}
df['Category2'] = df['Category'].replace(RARE_CLASS_MERGE)
print(df['Category2'].value_counts())

Category2
Retail Trade                                 13500
Entertainment                                11255
Trade, Professional and Personal Services     5275
Health and Community Services                 4157
Services to Transport                         2317
Travel                                        1489
Property and Business Services                1095
Other Services                                 912
Name: count, dtype: int64


## 5. Train / validation / test split (stratified 70/15/15)

In [6]:
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['Category2'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['Category2'], random_state=42)
print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

Train: (28000, 17) Val: (6000, 17) Test: (6000, 17)


## 6. Vectorize & train

In [7]:
vectorizer = TfidfVectorizer(max_features=6000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_train = vectorizer.fit_transform(train_df['clean_text'])
X_val = vectorizer.transform(val_df['clean_text'])
X_test = vectorizer.transform(test_df['clean_text'])

y_train, y_val, y_test = train_df['Category2'], val_df['Category2'], test_df['Category2']

clf = LogisticRegression(max_iter=1000, class_weight='balanced', C=5.0)
start = time.time()
clf.fit(X_train, y_train)
print(f"Trained in {time.time()-start:.1f}s")

Trained in 1.8s


## 7. Evaluation

In [8]:
val_preds = clf.predict(X_val)
print("VALIDATION")
print("Macro F1:", f1_score(y_val, val_preds, average='macro'))
print("Weighted F1:", f1_score(y_val, val_preds, average='weighted'))
print(classification_report(y_val, val_preds, zero_division=0))

VALIDATION
Macro F1: 0.5811387162019328
Weighted F1: 0.6780767650494345
                                           precision    recall  f1-score   support

                            Entertainment       0.81      0.76      0.79      1688
            Health and Community Services       0.78      0.76      0.77       624
                           Other Services       0.22      0.52      0.31       137
           Property and Business Services       0.22      0.55      0.32       164
                             Retail Trade       0.77      0.59      0.66      2025
                    Services to Transport       0.56      0.68      0.62       348
Trade, Professional and Personal Services       0.61      0.60      0.60       791
                                   Travel       0.48      0.72      0.58       223

                                 accuracy                           0.66      6000
                                macro avg       0.56      0.65      0.58      6000
             

In [9]:
test_preds = clf.predict(X_test)
print("HELD-OUT TEST")
print("Macro F1:", f1_score(y_test, test_preds, average='macro'))
print("Weighted F1:", f1_score(y_test, test_preds, average='weighted'))
print(classification_report(y_test, test_preds, zero_division=0))

HELD-OUT TEST
Macro F1: 0.5728042816500125
Weighted F1: 0.6742543586730012
                                           precision    recall  f1-score   support

                            Entertainment       0.81      0.76      0.78      1688
            Health and Community Services       0.81      0.78      0.79       623
                           Other Services       0.22      0.53      0.31       137
           Property and Business Services       0.20      0.49      0.28       164
                             Retail Trade       0.75      0.58      0.65      2025
                    Services to Transport       0.55      0.68      0.60       347
Trade, Professional and Personal Services       0.62      0.60      0.61       792
                                   Travel       0.45      0.68      0.54       224

                                 accuracy                           0.66      6000
                                macro avg       0.55      0.64      0.57      6000
          

## 8. Confidence-based fallback analysis

Rather than always returning a top-1 prediction, we check whether
withholding low-confidence predictions (flagging them "Needs Review"
instead) meaningfully improves reliability on the predictions we do keep.

In [10]:
probs = clf.predict_proba(X_test)
max_conf = probs.max(axis=1)
preds = clf.classes_[probs.argmax(axis=1)]

for t in [0.0, 0.3, 0.4, 0.5, 0.6]:
    accepted = max_conf >= t
    coverage = accepted.mean()
    acc = accuracy_score(np.array(y_test)[accepted], preds[accepted]) if accepted.sum() else float('nan')
    print(f"threshold={t:.1f} | coverage={coverage:.1%} | accuracy on accepted={acc:.3f}")

threshold=0.0 | coverage=100.0% | accuracy on accepted=0.659
threshold=0.3 | coverage=94.2% | accuracy on accepted=0.684
threshold=0.4 | coverage=80.9% | accuracy on accepted=0.747
threshold=0.5 | coverage=68.5% | accuracy on accepted=0.804
threshold=0.6 | coverage=58.9% | accuracy on accepted=0.856


**Decision: threshold = 0.4.** This auto-categorizes ~81% of transactions
at ~75% accuracy (vs. 66% with no fallback), routing the remaining ~19% to
manual review. See `docs/limitations_and_risks.md` for the full reasoning.

## 9. Export model for reuse and browser deployment

In [11]:
os.makedirs("model", exist_ok=True)
joblib.dump(vectorizer, "model/vectorizer_v2.joblib")
joblib.dump(clf, "model/clf_v2.joblib")

vocab = {k: int(v) for k, v in vectorizer.vocabulary_.items()}
export = {
    "vocab": vocab,
    "idf": [float(x) for x in vectorizer.idf_],
    "classes": [str(c) for c in clf.classes_],
    "coef": [[float(x) for x in row] for row in clf.coef_],
    "intercept": [float(x) for x in clf.intercept_],
    "ngram_range": [1, 2],
    "threshold": 0.4,
}
with open("model/model_export.json", "w") as f:
    json.dump(export, f)

print("Saved model artifacts to model/")

Saved model artifacts to model/


## 10. Try it interactively

The trained model is also embedded in `app/ledger.html` — a self-contained
web app with a single-transaction mode and a batch/statement mode, running
entirely in the browser via a JavaScript reimplementation of this same
TF-IDF + Logistic Regression pipeline (verified to produce identical
predictions to this notebook's Python model).

Open `app/ledger.html` directly in any browser to try it.